In [2]:
!pip -q install "transformers>=4.44" "trl>=0.9.4" "datasets>=2.19" "accelerate>=0.33" "chromadb>=0.5" "sentence-transformers>=5.1" "wandb>=0.23"

In [1]:
from prior_art_search.rollout import PatentSearchEnv, PatentSearchEnvConfig, PatentSearchGRPOTrainer, patent_reward_function

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig

DEFAULT_DATASET_PATH = Path("patent_search_queries.csv")


def assert_cuda_device() -> torch.device:
    """Require a CUDA device in Colab; raise with a helpful message otherwise."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    # If no GPU is present, fail fast with guidance.
    raise RuntimeError(
        "No CUDA device found. In Colab, set Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )


def load_dataset(csv_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    patent_search_queries = pd.read_csv(csv_path)
    train_df = patent_search_queries.sample(frac=0.8, random_state=42)
    val_df = patent_search_queries.drop(train_df.index)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def dataframe_to_dataset(df: pd.DataFrame) -> Dataset:
    records: List[Dict[str, object]] = []
    for row in df.to_dict(orient="records"):
        scenario = {
            "query": row.get("query", ""),
            "abstract": row.get("abstract", ""),
        }
        records.append(
            {
                "prompt": row.get("query", ""),
                "publication_number": str(row.get("publication_number", "")),
                "scenario": scenario,
            }
        )
    return Dataset.from_list(records)


def build_trainer(
    model_name: str,
    train_dataset: Dataset,
    eval_dataset: Dataset,
    max_steps: int,
) -> PatentSearchGRPOTrainer:
    device = assert_cuda_device()
    dtype = torch.float16

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map={"": device},
    )
    if getattr(model.config, "use_cache", True):
        model.config.use_cache = False

    env_config = PatentSearchEnvConfig(max_turns=6, turn_max_new_tokens=256, temperature=0.9, top_p=0.9)
    env = PatentSearchEnv(tokenizer=tokenizer, config=env_config)

    # Keep batches small for Colab GPUs; generation count must divide batch size.
    per_device_batch = 4
    rollouts_per_group = 4

    grpo_args = GRPOConfig(
        output_dir="/content/checkpoints/trl_patent_qwen",
        project="Agentic-Search-patent",
        run_name="Agentic-Search-patent-003",
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=per_device_batch,
        gradient_accumulation_steps=1,
        learning_rate=3e-4,
        max_prompt_length=2048,
        max_completion_length=1024,
        num_generations=rollouts_per_group,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=max_steps,
        max_steps=max_steps,
        report_to=["wandb"],
        save_steps=max_steps,
    )

    return PatentSearchGRPOTrainer(
        env=env,
        model=model,
        reward_funcs=patent_reward_function,
        args=grpo_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )



In [3]:
train_df, val_df = load_dataset(DEFAULT_DATASET_PATH)
train_dataset = dataframe_to_dataset(train_df)
eval_dataset = dataframe_to_dataset(val_df)

trainer = build_trainer(
        model_name="Qwen/Qwen3-0.6B",
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        max_steps=6,
    )

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [4]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: Currently logged in as: mounss (mounss-stude) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss


ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
/content/prior_art_search/rollout.py:207: RuntimeWarning: coroutine 'PatentSearchEnv._execute_tool.<locals>.runner' was never awaited
  break
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while another loop is running
ERROR:prior_art_search.rollout:Tool execution failed: Cannot run the event loop while an

KeyboardInterrupt: 